In [2]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import numpy as np

# =====================================================
# 1️⃣ Config
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Device: {device}")

data_dir = "data"
test_dir = os.path.join(data_dir, "seg_test", "seg_test")
model_dir = "models"

img_size = 224
batch_size = 32

# =====================================================
# 2️⃣ Transform + Load Test Data
# =====================================================
data_transforms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
class_names = test_dataset.classes
num_classes = len(class_names)
print(f"📂 Classes: {class_names}")

# =====================================================
# 3️⃣ Hàm build model giống lúc train
# =====================================================
def build_model(model_name="resnet50", num_classes=6):
    model_name = model_name.lower()

    if model_name == "vgg16":
        model = models.vgg16(weights=None)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "resnet50":
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "mobilenet":
        model = models.mobilenet_v2(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "efficientnet":
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "vit":
        model = vit_b_16(weights=None)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    else:
        raise ValueError(f"❌ Unknown model name: {model_name}")

    return model.to(device)

# =====================================================
# 4️⃣ Evaluation Function
# =====================================================
def evaluate_model(model, dataloader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    return acc, prec, rec, f1

# =====================================================
# 5️⃣ Load và Evaluate từng model
# =====================================================
model_files = {
    "vgg16": "vgg16_best.pt",
    "resnet50": "resnet50_best.pt",
    "mobilenet": "mobilenet_best.pt",
    "efficientnet": "efficientnet_best.pt",
    "vit": "vit_best.pt"
}

results = {}

for name, filename in model_files.items():
    print(f"\n============================")
    print(f"📦 Evaluating {name.upper()}")
    print(f"============================")

    model_path = os.path.join(model_dir, filename)
    model = build_model(name, num_classes=num_classes)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    acc, prec, rec, f1 = evaluate_model(model, test_loader)
    results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

# =====================================================
# 6️⃣ Print Results
# =====================================================
print("\n🏁 Final Evaluation Results:")
print("------------------------------------------------------")
for name, metrics in results.items():
    print(f"{name.upper():<12} | "
          f"Acc: {metrics['Accuracy']:.4f} | "
          f"Prec: {metrics['Precision']:.4f} | "
          f"Rec: {metrics['Recall']:.4f} | "
          f"F1: {metrics['F1']:.4f}")


🔥 Device: cuda
📂 Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

📦 Evaluating VGG16


C:\Users\TMS\AppData\Local\Temp\ipykernel_17288\4027662645.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de


📦 Evaluating RESNET50


C:\Users\TMS\AppData\Local\Temp\ipykernel_17288\4027662645.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de


📦 Evaluating MOBILENET


...


📦 Evaluating EFFICIENTNET


C:\Users\TMS\AppData\Local\Temp\ipykernel_17288\4027662645.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de


📦 Evaluating VIT


C:\Users\TMS\AppData\Local\Temp\ipykernel_17288\4027662645.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de


🏁 Final Evaluation Results:
------------------------------------------------------
VGG16        | Acc: 0.9290 | Prec: 0.9292 | Rec: 0.9290 | F1: 0.9288
RESNET50     | Acc: 0.9057 | Prec: 0.9055 | Rec: 0.9057 | F1: 0.9053
MOBILENET    | Acc: 0.8990 | Prec: 0.8986 | Rec: 0.8990 | F1: 0.8985
EFFICIENTNET | Acc: 0.8810 | Prec: 0.8810 | Rec: 0.8810 | F1: 0.8809
VIT          | Acc: 0.9297 | Prec: 0.9296 | Rec: 0.9297 | F1: 0.9294


In [3]:
import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights
import numpy as np

# =====================================================
# 1️⃣ Cấu hình cơ bản
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Device: {device}")

data_dir = "data"
test_dir = os.path.join(data_dir, "seg_test", "seg_test")
model_dir = "models"

img_size = 224
batch_size = 32

# =====================================================
# 2️⃣ Chuẩn hóa ảnh giống lúc huấn luyện
# =====================================================
data_transforms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load test dataset
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
num_classes = len(test_dataset.classes)
print(f"📂 Classes: {test_dataset.classes}")

# =====================================================
# 3️⃣ Hàm build model giống lúc train
# =====================================================
def build_model(model_name="resnet50", num_classes=6):
    model_name = model_name.lower()

    if model_name == "vgg16":
        model = models.vgg16(weights=None)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "resnet50":
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "mobilenet":
        model = models.mobilenet_v2(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "efficientnet":
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "vit":
        model = vit_b_16(weights=None)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    else:
        raise ValueError(f"❌ Unknown model name: {model_name}")

    return model.to(device)

# =====================================================
# 4️⃣ Hàm tính thời gian inference
# =====================================================
def measure_inference_time(model, dataloader, warmup_batches=2):
    model.eval()
    times = []

    with torch.no_grad():
        for i, (inputs, _) in enumerate(dataloader):
            inputs = inputs.to(device)

            if i < warmup_batches:
                _ = model(inputs)  # Warm-up (để GPU ổn định)
                continue

            start_time = time.time()
            _ = model(inputs)
            end_time = time.time()

            times.append(end_time - start_time)

    total_time = np.sum(times)
    avg_time_per_batch = np.mean(times)
    avg_time_per_image = avg_time_per_batch / dataloader.batch_size

    return total_time, avg_time_per_image

# =====================================================
# 5️⃣ Load model và đo thời gian
# =====================================================
model_files = {
    "vgg16": "vgg16_best.pt",
    "resnet50": "resnet50_best.pt",
    "mobilenet": "mobilenet_best.pt",
    "efficientnet": "efficientnet_best.pt",
    "vit": "vit_best.pt"
}

results = {}

for name, filename in model_files.items():
    print(f"\n============================")
    print(f"⏱️ Measuring Inference Time: {name.upper()}")
    print(f"============================")

    model_path = os.path.join(model_dir, filename)
    model = build_model(name, num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    total_time, avg_time = measure_inference_time(model, test_loader)

    results[name] = {
        "Total Time (s)": total_time,
        "Avg Time/Image (s)": avg_time
    }

# =====================================================
# 6️⃣ In kết quả
# =====================================================
print("\n🏁 Inference Time Results:")
print("------------------------------------------------------------")
for name, r in results.items():
    print(f"{name.upper():<12} | "
          f"Total: {r['Total Time (s)']:.2f}s | "
          f"Avg/Image: {r['Avg Time/Image (s)']*1000:.3f} ms")


🔥 Device: cuda
📂 Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

⏱️ Measuring Inference Time: VGG16


C:\Users\TMS\AppData\Local\Temp\ipykernel_17288\3131579208.py:147: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de


⏱️ Measuring Inference Time: RESNET50

⏱️ Measuring Inference Time: MOBILENET

⏱️ Measuring Inference Time: EFFICIENTNET

⏱️ Measuring Inference Time: VIT

🏁 Inference Time Results:
------------------------------------------------------------
VGG16        | Total: 0.44s | Avg/Image: 0.150 ms
RESNET50     | Total: 1.68s | Avg/Image: 0.570 ms
MOBILENET    | Total: 1.65s | Avg/Image: 0.561 ms
EFFICIENTNET | Total: 2.63s | Avg/Image: 0.895 ms
VIT          | Total: 1.67s | Avg/Image: 0.566 ms


In [4]:
import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights
import numpy as np

# =====================================================
# 1️⃣ Cấu hình cơ bản
# =====================================================
device = torch.device("cpu")
print(f"🔥 Device: {device}")

data_dir = "data"
test_dir = os.path.join(data_dir, "seg_test", "seg_test")
model_dir = "models"

img_size = 224
batch_size = 32

# =====================================================
# 2️⃣ Chuẩn hóa ảnh giống lúc huấn luyện
# =====================================================
data_transforms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load test dataset
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
num_classes = len(test_dataset.classes)
print(f"📂 Classes: {test_dataset.classes}")

# =====================================================
# 3️⃣ Hàm build model giống lúc train
# =====================================================
def build_model(model_name="resnet50", num_classes=6):
    model_name = model_name.lower()

    if model_name == "vgg16":
        model = models.vgg16(weights=None)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "resnet50":
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "mobilenet":
        model = models.mobilenet_v2(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "efficientnet":
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    elif model_name == "vit":
        model = vit_b_16(weights=None)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    else:
        raise ValueError(f"❌ Unknown model name: {model_name}")

    return model.to(device)

# =====================================================
# 4️⃣ Hàm tính thời gian inference
# =====================================================
def measure_inference_time(model, dataloader, warmup_batches=2):
    model.eval()
    times = []

    with torch.no_grad():
        for i, (inputs, _) in enumerate(dataloader):
            inputs = inputs.to(device)

            if i < warmup_batches:
                _ = model(inputs)  # Warm-up (để GPU ổn định)
                continue

            start_time = time.time()
            _ = model(inputs)
            end_time = time.time()

            times.append(end_time - start_time)

    total_time = np.sum(times)
    avg_time_per_batch = np.mean(times)
    avg_time_per_image = avg_time_per_batch / dataloader.batch_size

    return total_time, avg_time_per_image

# =====================================================
# 5️⃣ Load model và đo thời gian
# =====================================================
model_files = {
    "vgg16": "vgg16_best.pt",
    "resnet50": "resnet50_best.pt",
    "mobilenet": "mobilenet_best.pt",
    "efficientnet": "efficientnet_best.pt",
    "vit": "vit_best.pt"
}

results = {}

for name, filename in model_files.items():
    print(f"\n============================")
    print(f"⏱️ Measuring Inference Time: {name.upper()}")
    print(f"============================")

    model_path = os.path.join(model_dir, filename)
    model = build_model(name, num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    total_time, avg_time = measure_inference_time(model, test_loader)

    results[name] = {
        "Total Time (s)": total_time,
        "Avg Time/Image (s)": avg_time
    }

# =====================================================
# 6️⃣ In kết quả
# =====================================================
print("\n🏁 Inference Time Results:")
print("------------------------------------------------------------")
for name, r in results.items():
    print(f"{name.upper():<12} | "
          f"Total: {r['Total Time (s)']:.2f}s | "
          f"Avg/Image: {r['Avg Time/Image (s)']*1000:.3f} ms")


🔥 Device: cpu
📂 Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

⏱️ Measuring Inference Time: VGG16


C:\Users\TMS\AppData\Local\Temp\ipykernel_17288\3557058552.py:147: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=de


⏱️ Measuring Inference Time: RESNET50

⏱️ Measuring Inference Time: MOBILENET

⏱️ Measuring Inference Time: EFFICIENTNET

⏱️ Measuring Inference Time: VIT

🏁 Inference Time Results:
------------------------------------------------------------
VGG16        | Total: 503.46s | Avg/Image: 171.012 ms
RESNET50     | Total: 162.50s | Avg/Image: 55.195 ms
MOBILENET    | Total: 38.60s | Avg/Image: 13.110 ms
EFFICIENTNET | Total: 55.52s | Avg/Image: 18.860 ms
VIT          | Total: 556.90s | Avg/Image: 189.163 ms
